# 🧠 Children's Story Quiz Generator
## Seq2Seq Deep Learning — Flan-T5 Fine-Tuning

---
> **Architecture:** Encoder-Decoder (Flan-T5) — completely different from Story Lab
> **Model:** `google/flan-t5-base` (250M params)
> **Training:** Full Seq2Seq fine-tuning with `Seq2SeqTrainer`
> **Generation:** Beam Search (not sampling)
> **Key idea:** Encoder reads the FULL story, Decoder generates the quiz
---

## Table of Contents
1. [Setup](#section1)
2. [Dataset Generation with Groq](#section2)
3. [Data Preparation for Seq2Seq](#section3)
4. [Flan-T5 Model Loading](#section4)
5. [Seq2Seq Fine-Tuning](#section5)
6. [Save and Load](#section6)
7. [Full Pipeline: Story to Quiz](#section7)
8. [Evaluation](#section8)
9. [Architecture Comparison](#section9)
10. [Interactive Demo](#section10)


---
<a id='section1'></a>
## Section 1 — Setup and Dependencies


In [ ]:
import subprocess, sys

packages = [
    'groq>=0.9.0',
    'torch',
    'transformers>=4.38.0',
    'datasets',
    'accelerate',
    'sentencepiece',
    'protobuf',
    'evaluate',
    'rouge-score',
    'nltk',
    'sentence-transformers',
    'scikit-learn',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('All packages installed.')


KeyboardInterrupt: 

In [2]:
import os, json, re, time, math, random, warnings
warnings.filterwarnings('ignore')
import torch
import numpy as np
import nltk
nltk.download('punkt', quiet=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE.upper()}')
print(f'PyTorch : {torch.__version__}')

os.makedirs('data', exist_ok=True)
os.makedirs('outputs/t5-quiz', exist_ok=True)
print('Directories ready')


KeyboardInterrupt: 

---
<a id='section2'></a>
## Section 2 — Dataset Generation with Groq

### Quiz Output Format
T5 outputs **structured plain text** with `###` separators for reliable parsing:

```
Q1: What is the rabbit's name?
A) Sunny B) Pip C) Luna D) Milo
Answer: B
Explanation: The story says a little rabbit named Pip.
###
Q2: Where does the story take place?
A) Ocean B) Forest C) Meadow D) Mountain
Answer: C
Explanation: The story begins in a golden meadow.
```


In [ ]:
from groq import Groq

GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')
if not GROQ_API_KEY:
    raise ValueError('Please set GROQ_API_KEY in your environment or Colab Secrets.')

client     = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL = 'llama-3.1-8b-instant'
print(f'Groq client ready — model: {GROQ_MODEL}')


In [ ]:
import os, json, re, time
os.makedirs('data', exist_ok=True)

STORY_PATH = 'data/caption_stories.jsonl'

# Generate fresh stories using Groq
STORY_SYSTEM = (
    "You are a children's book author. Write warm, imaginative, "
    "safe stories for children aged 4-8."
)

STORY_TEMPLATE = (
    "Write a children's story of EXACTLY 20 lines based on this caption.\n\n"
    "Caption: {caption}\n\n"
    "Rules:\n"
    "- EXACTLY 20 lines, each a complete sentence\n"
    "- Simple warm English for children aged 4-8\n"
    "- No violence or adult themes\n\n"
    "Output the 20 lines directly, one per line. No numbering."
)

CAPTIONS = [
    "A small rabbit sits beside a sunflower in a golden meadow.",
    "A baby elephant splashes happily in a shallow blue river.",
    "A tiny owl perches on a moonlit branch surrounded by fireflies.",
    "A little girl blows dandelion seeds into a warm summer breeze.",
    "A dragon with tiny wings sits on a mountain blowing smoke rings.",
    "A baby penguin stands on an iceberg watching colourful fish.",
    "Two children fly a bright red kite on top of a green hill.",
    "A fairy sleeps inside a foxglove flower in the forest.",
    "A child looks through a telescope at a sky full of stars.",
    "A little astronaut bounces across the surface of the moon.",
    "A sea turtle glides peacefully through clear blue water.",
    "A golden puppy chases soap bubbles in a sunny garden.",
    "A small unicorn grazes beside a crystal stream.",
    "A child builds a sandcastle at the edge of the ocean.",
    "Cherry blossom petals drift over a quiet stone bridge.",
    "A lighthouse glows on a rocky shore as waves crash below.",
    "A gnome hangs a lantern outside his mushroom house.",
    "A dolphin leaps over the bow of a little wooden sailboat.",
    "A paper lantern floats into a night sky full of stars.",
    "A family of ducks waddles across a quiet country road.",
]

print(f"Generating {len(CAPTIONS)} stories with Groq...\n")

existing = []
if os.path.exists(STORY_PATH):
    with open(STORY_PATH, 'r') as f:
        existing = [json.loads(l) for l in f if l.strip()]
    print(f"Found {len(existing)} existing stories, resuming...")

existing_caps = {s['caption'] for s in existing}
new_stories = []

story_file = open(STORY_PATH, 'a', encoding='utf-8')
try:
    for idx, caption in enumerate(CAPTIONS, 1):
        if caption in existing_caps:
            print(f"  [{idx:02d}] Skip (exists)", flush=True)
            continue
        print(f"  [{idx:02d}/{len(CAPTIONS)}] {caption[:50]}...", flush=True)
        try:
            resp = client.chat.completions.create(
                model=GROQ_MODEL,
                max_tokens=600,
                temperature=0.75,
                messages=[
                    {"role": "system", "content": STORY_SYSTEM},
                    {"role": "user",   "content": STORY_TEMPLATE.format(caption=caption)},
                ]
            )
            raw   = resp.choices[0].message.content.strip()
            lines = [re.sub(r"^\d+[\.\)]\s*", "", l.strip())
                     for l in raw.split("\n") if l.strip()]
            if len(lines) >= 15:
                story  = "\n".join(lines[:20])
                sample = {"caption": caption, "story": story}
                new_stories.append(sample)
                story_file.write(json.dumps(sample, ensure_ascii=False) + "\n")
                story_file.flush()
                print(f"           {len(lines)} lines", flush=True)
        except Exception as e:
            print(f"           Error: {e}", flush=True)
        time.sleep(0.5)
except KeyboardInterrupt:
    print("Stopped.")
finally:
    story_file.close()

# Load final dataset
with open(STORY_PATH, 'r', encoding='utf-8') as f:
    ALL_STORIES = [json.loads(l) for l in f if l.strip()]

print(f"\nLoaded {len(ALL_STORIES)} stories total")
print(f"Example: {ALL_STORIES[0]['caption']}")

In [ ]:
QUIZ_SYSTEM = (
    'You are an expert quiz designer for children aged 4-8. '
    'You create multiple choice comprehension questions. '
    'You always follow the exact output format.'
)

QUIZ_TEMPLATE = (
    'Read this children story and write exactly 5 multiple choice questions.\n\n'
    'Story:\n{story}\n\n'
    'Rules:\n'
    '- 5 questions: character, setting, event, emotion, moral\n'
    '- 4 options on ONE line: A) opt B) opt C) opt D) opt\n'
    '- Answer: one letter A B C or D\n'
    '- Explanation: one sentence\n'
    '- Separate with ### between questions\n\n'
    'Format:\n'
    'Q1: question\n'
    'A) opt B) opt C) opt D) opt\n'
    'Answer: X\n'
    'Explanation: sentence\n'
    '###\n'
    'Q2: question\n'
    '...up to Q5 with no ### after last'
)

print('Prompts ready')


In [ ]:
def parse_wait_seconds(err):
    m = re.search(r'(\d+)m([\d.]+)s', str(err))
    if m: return int(m.group(1)) * 60 + float(m.group(2))
    m = re.search(r'([\d.]+)s', str(err))
    if m: return float(m.group(1))
    return 60


def generate_quiz_groq(story, max_retries=3):
    attempt = 0
    rate_retries = 0
    while attempt < max_retries:
        try:
            resp = client.chat.completions.create(
                model=GROQ_MODEL,
                max_tokens=1000,
                temperature=0.4,
                messages=[
                    {'role': 'system', 'content': QUIZ_SYSTEM},
                    {'role': 'user',   'content': QUIZ_TEMPLATE.format(story=story)},
                ]
            )
            raw = resp.choices[0].message.content.strip()
            q_n = len(re.findall(r'Q\d+:', raw))
            a_n = len(re.findall(r'Answer:\s*[ABCD]', raw))
            if q_n >= 4 and a_n >= 4:
                return raw
            print(f'    Only {q_n} questions, retrying...', flush=True)
            attempt += 1
        except Exception as e:
            err_str = str(e)
            if '429' in err_str:
                wait = parse_wait_seconds(err_str)
                rate_retries += 1
                if rate_retries > 8: return None
                print(f'    Rate limit — waiting {wait:.0f}s [{rate_retries}/8]...', flush=True)
                total = int(wait) + 5
                for i in range(total):
                    time.sleep(1)
                    if i % 20 == 0 and i > 0:
                        print(f'      {total-i}s left...', flush=True)
                print('    Retrying...', flush=True)
            else:
                print(f'    Error: {e}', flush=True)
                time.sleep(2 ** attempt)
                attempt += 1
    return None


print('Smoke test...', flush=True)
_quiz = generate_quiz_groq(ALL_STORIES[0]['story'])
if _quiz:
    print(f'Passed: {len(re.findall(r"Q\\d+:", _quiz))} questions')
    print('Preview:\n' + _quiz[:400] + '...')
else:
    print('Failed — check API key')


In [ ]:
QUIZ_PATH  = 'data/story_quiz_pairs.jsonl'
DELAY_SECS = 0.5

existing = []
if os.path.exists(QUIZ_PATH):
    with open(QUIZ_PATH, 'r', encoding='utf-8') as f:
        existing = [json.loads(l) for l in f if l.strip()]
    print(f'Resuming: {len(existing)} pairs on disk.', flush=True)

existing_caps = {s['caption'] for s in existing}
new_pairs, failed = [], []
total = len(ALL_STORIES)

print(f'Total   : {total}', flush=True)
print(f'Done    : {len(existing_caps)}', flush=True)
print(f'Left    : {total - len(existing_caps)}\n', flush=True)

qfile = open(QUIZ_PATH, 'a', encoding='utf-8')
try:
    for idx, s in enumerate(ALL_STORIES, 1):
        if s['caption'] in existing_caps:
            continue
        print(f'  [{idx:03d}/{total}] {s["caption"][:55]}...', flush=True)
        quiz = generate_quiz_groq(s['story'])
        if quiz:
            pair = {'caption': s['caption'], 'story': s['story'], 'quiz': quiz}
            new_pairs.append(pair)
            qfile.write(json.dumps(pair, ensure_ascii=False) + '\n')
            qfile.flush()
            print(f'           {len(re.findall(r"Q\\d+:", quiz))} questions', flush=True)
        else:
            failed.append(s['caption'])
            print('           FAILED', flush=True)
        time.sleep(DELAY_SECS)
        done = len(existing_caps) + len(new_pairs)
        if len(new_pairs) % 25 == 0 and len(new_pairs) > 0:
            print(f'\n  Progress: {done}/{total} ({done/total*100:.0f}%)\n', flush=True)
except KeyboardInterrupt:
    print('\nStopped — progress saved.', flush=True)
finally:
    qfile.close()

total_saved = len(existing) + len(new_pairs)
print('\n' + '='*50)
print(f'Generated : {len(new_pairs)}')
print(f'Failed    : {len(failed)}')
print(f'Total     : {total_saved}')


In [ ]:
with open(QUIZ_PATH, 'r', encoding='utf-8') as f:
    QUIZ_DATASET = [json.loads(l) for l in f if l.strip()]

q_counts = [len(re.findall(r'Q\d+:', s['quiz'])) for s in QUIZ_DATASET]
print(f'Total pairs         : {len(QUIZ_DATASET)}')
print(f'Avg questions/quiz  : {sum(q_counts)/len(q_counts):.1f}')
print(f'Full quizzes (5 Qs) : {sum(1 for n in q_counts if n >= 5)}')
print('\nSample:\n' + QUIZ_DATASET[0]['quiz'][:400] + '...')


---
<a id='section3'></a>
## Section 3 — Data Preparation for Seq2Seq

### Key Difference from Story Lab

| Story Lab (Causal LM) | Quiz Lab (Seq2Seq) |
|----------------------|-------------------|
| One long string: `caption + story` | **Two separate strings**: input and target |
| Model predicts next token | Model maps input to output sequence |
| `input_ids` only | `input_ids` + `labels` (separate tensors) |
| SFTTrainer | **Seq2SeqTrainer** |
| Loss over all tokens | Loss over **target tokens only** |

```python
# Story Lab:
{'text': 'Caption: A rabbit...  Story: Once upon...'}

# Quiz Lab:
{'input_text':  'Generate quiz: Once upon a time...',
 'target_text': 'Q1: What is... Answer: B...'}
```


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME     = 'google/flan-t5-base'
MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 256

print('Loading Flan-T5 tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded — vocab size: {tokenizer.vocab_size:,}')
print(f'Type: {tokenizer.__class__.__name__}')


In [ ]:
def make_input_text(story):
    return 'Generate a 5-question multiple choice quiz for this children story: ' + story


def tokenize_function(examples):
    model_inputs = tokenizer(
        examples['input_text'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding='max_length',
    )
    labels = tokenizer(
        text_target=examples['target_text'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding='max_length',
    )
    # Mask padding tokens — loss ignores them
    label_ids = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in lbl]
        for lbl in labels['input_ids']
    ]
    model_inputs['labels'] = label_ids
    return model_inputs


sample_input  = make_input_text(QUIZ_DATASET[0]['story'])
sample_target = QUIZ_DATASET[0]['quiz']
print('INPUT (first 200 chars):')
print(sample_input[:200])
print('\nTARGET (first 200 chars):')
print(sample_target[:200])
print(f'\nInput tokens  : ~{len(tokenizer.encode(sample_input))}')
print(f'Target tokens : ~{len(tokenizer.encode(sample_target))}')


In [ ]:
import random
random.seed(42)
shuffled = QUIZ_DATASET[:]
random.shuffle(shuffled)

split_idx  = max(1, int(len(shuffled) * 0.85))
TRAIN_DATA = shuffled[:split_idx]
VAL_DATA   = shuffled[split_idx:]

def build_dataset(data):
    return Dataset.from_list([
        {'input_text' : make_input_text(s['story']),
         'target_text': s['quiz'],
         'caption'    : s['caption']}
        for s in data
    ])

raw_train = build_dataset(TRAIN_DATA)
raw_val   = build_dataset(VAL_DATA)

print('Tokenizing...')
train_dataset = raw_train.map(
    tokenize_function, batched=True,
    remove_columns=['input_text', 'target_text', 'caption']
)
val_dataset = raw_val.map(
    tokenize_function, batched=True,
    remove_columns=['input_text', 'target_text', 'caption']
)

train_dataset.set_format('torch')
val_dataset.set_format('torch')

print(f'Train : {len(train_dataset)} samples')
print(f'Val   : {len(val_dataset)} samples')
print(f'Columns: {train_dataset.column_names}')


---
<a id='section4'></a>
## Section 4 — Flan-T5 Model Loading

| Property | Flan-T5-Base | TinyLlama (Story Lab) |
|----------|-------------|---------------------|
| **Architecture** | Encoder-Decoder | Decoder-only |
| **Parameters** | 250M | 1.1B |
| **Pre-training** | Instruction-tuned (1800+ tasks) | Next-token prediction |
| **Fine-tuning** | Full (no LoRA needed) | LoRA (PEFT) |
| **Reading** | Full context at once | Left to right |
| **Generation** | Beam Search | Sampling |

Flan-T5 already understands 'Generate a quiz' — instruction pre-training makes fine-tuning much faster.


In [ ]:
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

OUTPUT_DIR = 'outputs/t5-quiz'

print('Loading Flan-T5-Base...')
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model              : {MODEL_NAME}')
print(f'Total parameters   : {total/1e6:.0f}M')
print(f'Trainable params   : {trainable/1e6:.0f}M  (100% full fine-tuning — no LoRA needed)')
print(f'Architecture       : Encoder-Decoder (T5)')
print(f'Device             : {next(model.parameters()).device}')


In [ ]:
# DataCollatorForSeq2Seq pads each batch dynamically
# More efficient than fixed padding used in Story Lab
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

print('DataCollatorForSeq2Seq ready')
print('Dynamic padding per batch — more efficient than Story Lab fixed padding')


---
<a id='section5'></a>
## Section 5 — Seq2Seq Fine-Tuning

### Key Differences from Story Lab

| | Story Lab | Quiz Lab |
|-|-----------|----------|
| Trainer | SFTTrainer | **Seq2SeqTrainer** |
| Loss | All tokens | **Target tokens only** |
| Generation at eval | Sampling | **Beam search (4 beams)** |
| Best metric | eval_loss | **ROUGE-L** |
| Epochs | 3 | 5 (T5 learns faster) |

Beam search generates **4 quiz candidates** simultaneously and picks the best — ideal for structured MCQ output.


In [ ]:
import evaluate as hf_evaluate
import numpy as np

rouge = hf_evaluate.load('rouge')


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    # ROUGE score
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    # Answer accuracy: do generated A/B/C/D match reference?
    correct = 0
    total   = 0
    for pred, label in zip(decoded_preds, decoded_labels):
        pred_ans  = re.findall(r'Answer:\s*([ABCD])', pred)
        label_ans = re.findall(r'Answer:\s*([ABCD])', label)
        for p, l in zip(pred_ans, label_ans):
            total += 1
            if p == l: correct += 1

    result['answer_accuracy'] = correct / total if total > 0 else 0.0
    return {k: round(v, 4) for k, v in result.items()}


print('compute_metrics() ready')
print('Tracks: ROUGE-1, ROUGE-2, ROUGE-L, answer_accuracy')


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Core
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,    # effective batch = 8
    warmup_steps=50,

    # Optimiser
    learning_rate=3e-4,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    max_grad_norm=1.0,

    # Seq2Seq specific
    predict_with_generate=True,       # use beam search for evaluation
    generation_max_length=256,
    generation_num_beams=4,           # beam width

    # Precision
    fp16=torch.cuda.is_available(),
    bf16=False,

    # Logging
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    greater_is_better=True,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,   # ← was: tokenizer=tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Seq2SeqTrainer ready')
print(f'Train : {len(train_dataset)} | Val : {len(val_dataset)}')
print(f'Epochs: 5 | Beam search: 4 | Effective batch: 8')

In [ ]:
print('Starting Seq2Seq fine-tuning...\n')
train_result = trainer.train()

print(f'\nTraining complete!')
print(f'Loss    : {train_result.training_loss:.4f}')
print(f'Runtime : {train_result.metrics.get("train_runtime", 0):.1f}s')

print('\nFinal evaluation with beam search...')
eval_results = trainer.evaluate(max_length=256, num_beams=4)

print('\nEvaluation Results:')
for k, v in eval_results.items():
    if isinstance(v, float):
        print(f'  {k:<28}: {v:.4f}')


---
<a id='section6'></a>
## Section 6 — Save and Load the Model

With T5 we save the **full fine-tuned model** (not just LoRA adapters).
Flan-T5-Base is about 1 GB on disk.


In [ ]:
SAVE_PATH = os.path.join(OUTPUT_DIR, 'flan_t5_quiz_model')
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

total_size = sum(
    os.path.getsize(os.path.join(SAVE_PATH, f))
    for f in os.listdir(SAVE_PATH)
)
print(f'Model saved to : {SAVE_PATH}')
print(f'Total size     : {total_size/1024/1024:.0f} MB')
for fn in sorted(os.listdir(SAVE_PATH)):
    mb = os.path.getsize(os.path.join(SAVE_PATH, fn)) / 1024 / 1024
    print(f'  {fn:<40}  {mb:6.1f} MB')


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

def load_quiz_model(path):
    tok = AutoTokenizer.from_pretrained(path)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(path).to(DEVICE)
    mdl.eval()
    return mdl, tok

quiz_model, quiz_tokenizer = load_quiz_model(SAVE_PATH)
print(f'Model loaded from {SAVE_PATH}')
print(f'Device: {next(quiz_model.parameters()).device}')


---
<a id='section7'></a>
## Section 7 — Full Pipeline: Story to Quiz

```
Image -> Caption Model -> Story Model (TinyLlama) -> Quiz Model (Flan-T5) -> 5 MCQ
```

Flan-T5 uses **beam search** — generates 4 quiz candidates and picks the best one.


In [ ]:
def generate_quiz_t5(story, model, tokenizer, num_beams=4, max_new_tokens=256):
    input_text = make_input_text(story)
    inputs = tokenizer(
        input_text,
        return_tensors='pt',
        max_length=MAX_INPUT_LEN,
        truncation=True,
    ).to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=3,
            length_penalty=1.5,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


def parse_quiz_t5(raw):
    questions = []
    blocks = re.split(r'\n?###\n?', raw.strip())
    for block in blocks:
        lines = [l.strip() for l in block.strip().split('\n') if l.strip()]
        if not lines: continue
        q = {}
        for line in lines:
            if re.match(r'Q\d+:', line):
                q['question'] = re.sub(r'^Q\d+:\s*', '', line)
            elif re.match(r'A\)', line):
                opts = re.findall(r'([ABCD])\)\s*(.*?)(?=\s[ABCD]\)|$)', line)
                for letter, text in opts:
                    q[letter] = text.strip()
            elif line.startswith('Answer:'):
                q['answer'] = line.replace('Answer:', '').strip()[:1]
            elif line.startswith('Explanation:'):
                q['explanation'] = line.replace('Explanation:', '').strip()
        if 'question' in q and 'answer' in q:
            questions.append(q)
    return questions


def display_quiz_t5(caption, raw):
    questions = parse_quiz_t5(raw)
    print('=' * 70)
    print(f'Story: {caption}')
    print('=' * 70)
    if not questions:
        print('Raw output:\n' + raw)
        return
    print(f'{len(questions)} Questions\n')
    for i, q in enumerate(questions, 1):
        print(f'Q{i}: {q.get("question", "N/A")}')
        for opt in ['A', 'B', 'C', 'D']:
            marker = '[correct]' if opt == q.get('answer', '') else '         '
            print(f'  {marker} {opt}) {q.get(opt, "N/A")}')
        print(f'   Explanation: {q.get("explanation", "")}')
        print()
    print('=' * 70)


print('generate_quiz_t5(), parse_quiz_t5(), display_quiz_t5() ready')


In [ ]:
# Test on 3 validation stories
print('Testing on 3 validation stories...\n')
for sample in VAL_DATA[:3]:
    raw = generate_quiz_t5(sample['story'], quiz_model, quiz_tokenizer)
    display_quiz_t5(sample['caption'], raw)
    print()


---
<a id='section8'></a>
## Section 8 — Evaluation

| Metric | What it measures | Target |
|--------|-----------------|--------|
| **Answer Accuracy** | % of A/B/C/D matching reference | > 70% |
| **Question Count** | Questions generated (target: 5) | 5 |
| **ROUGE-L** | Text overlap with reference quiz | > 0.30 |
| **Semantic Similarity** | Meaning alignment | > 0.70 |
| **Exact Match** | Full question matches reference | bonus |


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
import evaluate as hf_evaluate

rouge_metric = hf_evaluate.load('rouge')
print('Loading sentence embedder...')
embed_model  = SentenceTransformer('all-MiniLM-L6-v2')
print('All evaluation tools ready')


def full_eval(story, ref_quiz, gen_quiz):
    # Answer accuracy
    ref_ans = re.findall(r'Answer:\s*([ABCD])', ref_quiz)
    gen_ans = re.findall(r'Answer:\s*([ABCD])', gen_quiz)
    correct = sum(p == l for p, l in zip(gen_ans, ref_ans))
    ans_acc = correct / len(ref_ans) if ref_ans else 0

    # Question count
    q_count = len(re.findall(r'Q\d+:', gen_quiz))

    # ROUGE-L
    rouge_r = rouge_metric.compute(predictions=[gen_quiz], references=[ref_quiz])

    # Semantic similarity
    e1  = embed_model.encode([ref_quiz])
    e2  = embed_model.encode([gen_quiz])
    sem = float(cos_sim(e1, e2)[0][0])

    # Exact match on questions
    ref_qs = re.findall(r'Q\d+:\s*(.+)', ref_quiz)
    gen_qs = re.findall(r'Q\d+:\s*(.+)', gen_quiz)
    exact  = sum(r.strip() == g.strip() for r, g in zip(ref_qs, gen_qs))
    em     = exact / len(ref_qs) if ref_qs else 0

    return {
        'answer_accuracy': round(ans_acc, 4),
        'question_count' : q_count,
        'rouge_l'        : round(rouge_r['rougeL'], 4),
        'semantic_sim'   : round(sem, 4),
        'exact_match'    : round(em, 4),
    }

print('full_eval() ready')


In [ ]:
print('Evaluating on validation set...\n')
print(f'{"#":<4} {"Ans%":>6} {"Qs":>3} {"ROUGE-L":>8} {"Sem.Sim":>8} {"ExactM":>7}  Caption')
print('-' * 78)

all_m = []
for i, sample in enumerate(VAL_DATA, 1):
    gen = generate_quiz_t5(sample['story'], quiz_model, quiz_tokenizer)
    m   = full_eval(sample['story'], sample['quiz'], gen)
    all_m.append(m)
    cap = (sample['caption'][:30] + '...') if len(sample['caption']) > 30 else sample['caption']
    print(f'{i:<4} {m["answer_accuracy"]*100:>5.0f}%'
          f' {m["question_count"]:>3}'
          f' {m["rouge_l"]:>8.4f}'
          f' {m["semantic_sim"]:>8.4f}'
          f' {m["exact_match"]:>7.4f}'
          f'  {cap}')

n = len(VAL_DATA)
print('-' * 78)
print(f'{"AVG":<4}'
      f' {sum(m["answer_accuracy"] for m in all_m)/n*100:>5.0f}%'
      f' {sum(m["question_count"] for m in all_m)/n:>3.1f}'
      f' {sum(m["rouge_l"] for m in all_m)/n:>8.4f}'
      f' {sum(m["semantic_sim"] for m in all_m)/n:>8.4f}'
      f' {sum(m["exact_match"] for m in all_m)/n:>7.4f}')

print('\nInterpretation:')
print('  Ans%    > 70%   -- correct answer letters')
print('  Qs      = 5     -- complete quiz')
print('  ROUGE-L > 0.30  -- good text overlap')
print('  Sem.Sim > 0.70  -- strong meaning alignment')


---
<a id='section9'></a>
## Section 9 — Architecture Comparison: Flan-T5 vs TinyLlama


In [ ]:
print('='*65)
print('HOW EACH MODEL READS AND GENERATES')
print('='*65)
print()
print('STORY LAB — TinyLlama (Decoder-only)')
print('  Input : [Caption] -> [Story line 1] -> [line 2] -> ...')
print('  Reads : left to right, one token at a time')
print('  Output: next token predicted at each step')
print('  Loss  : over ALL tokens (caption + story)')
print()
print('QUIZ LAB — Flan-T5 (Encoder-Decoder)')
print('  ENCODER: reads FULL story with bidirectional attention')
print('           builds rich understanding of the whole story')
print('  DECODER: generates quiz token by token')
print('           attends to full story at every generation step')
print('  Loss  : over TARGET tokens only (quiz)')
print()
print('='*65)
print('COMPARISON TABLE')
print('='*65)
rows = [
    ('Parameters',    '1.1B',           '250M'),
    ('Architecture',  'Decoder-only',   'Encoder-Decoder'),
    ('Fine-tuning',   'LoRA (2M)',       'Full (250M)'),
    ('Trainer',       'SFTTrainer',      'Seq2SeqTrainer'),
    ('Data format',   'Single sequence', 'Input + Target'),
    ('Generation',    'Sampling',        'Beam Search x4'),
    ('Best metric',   'Perplexity',      'ROUGE-L + Ans Acc'),
    ('Context',       'Left-to-right',   'Full bidirectional'),
    ('Speed',         'Slower (bigger)', 'Faster (smaller)'),
]
print(f'{"Property":<20} {"Story Lab (TinyLlama)":^24} {"Quiz Lab (Flan-T5)":^22}')
print('-' * 68)
for prop, tiny, t5 in rows:
    print(f'{prop:<20} {tiny:^24} {t5:^22}')


---
<a id='section10'></a>
## Section 10 — Interactive Demo


In [ ]:
YOUR_STORY = '\n'.join([
    'Once upon a time a little blue fish named Finn lived in a coral reef.',
    'Finn had shiny scales that sparkled like diamonds in the sunlight.',
    'Every morning he swam around the reef looking for his breakfast.',
    'His favourite food was tiny pink shrimp that hid behind the coral.',
    'One day Finn found a strange glowing shell at the bottom of the sea.',
    'He touched it gently with his fin and it opened with a soft click.',
    'Inside was the most beautiful pearl Finn had ever seen.',
    'It glowed with warm golden light that lit up the whole reef.',
    'A wise old sea turtle swam over and looked at the pearl kindly.',
    'That pearl grants one wish to whoever finds it said the turtle.',
    'Finn thought very carefully about what he wanted most in the world.',
    'He wished for all the coral in the reef to grow back strong and bright.',
    'Suddenly the water filled with colour as new coral bloomed everywhere.',
    'Fish and starfish and crabs came out to dance in the new reef garden.',
    'Finn felt so happy that his heart glowed just like the golden pearl.',
    'He placed the shell gently back for the next finder to discover.',
    'The old turtle smiled and nodded his head with deep approval.',
    'That was a very wise and kind wish he said swimming slowly away.',
    'Finn swam home through the most beautiful reef he had ever seen.',
    'And from that day on the little blue fish was the happiest fish in the sea.',
])

print('Generating quiz with Flan-T5 beam search...\n')
raw = generate_quiz_t5(YOUR_STORY, quiz_model, quiz_tokenizer)
display_quiz_t5('A little blue fish named Finn finds a magical pearl', raw)


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('flan_t5_quiz_model', 'zip', SAVE_PATH)
size_mb = os.path.getsize('flan_t5_quiz_model.zip') / 1024 / 1024
print(f'Model zipped: {size_mb:.0f} MB')
files.download('flan_t5_quiz_model.zip')


---

## Lab Complete!

### Full End-to-End System

```
Image
  |
  v
Caption Model (BLIP)
  |
  v
Story Model  -- TinyLlama 1.1B + LoRA  -- Decoder-only  -- Causal LM
  |
  v
Quiz Model   -- Flan-T5-Base 250M      -- Encoder-Decoder -- Seq2Seq
  |
  v
5 Multiple Choice Questions with Answers
```

### Technologies Comparison

| | Story Lab | Quiz Lab |
|-|-----------|----------|
| Model | TinyLlama 1.1B | Flan-T5-Base 250M |
| Architecture | Decoder-only | Encoder-Decoder |
| Fine-tuning | LoRA (PEFT) | Full fine-tuning |
| Trainer | SFTTrainer | Seq2SeqTrainer |
| Generation | Temperature sampling | Beam Search |
| Data format | Single sequence | Input + Target |
| Key metric | Perplexity | Answer Accuracy |

### Next Steps
- Use `flan-t5-large` (780M) for better quality
- Add difficulty levels: easy / medium / hard
- Build a Gradio web app for teachers
- Export quizzes as PDF worksheets
